# Dynamic Prototype FSL: Few-Shot Learning with Dynamic Prototype Refinement
===========================================================================
**Author:** NAYANCSE27  
**Research Level Implementation**

This notebook implements **Dynamic Prototype Methods** for Few-Shot Learning integrated with Explainable AI (XAI) techniques.

## Table of Contents
1. [Introduction & Mathematical Framework](#1-introduction)
2. [Dataset Setup & Stratified Splitting](#2-dataset)
3. [Model Architecture](#3-model)
4. [Dynamic Prototype Refinement](#4-dynamic-prototype)
5. [XAI Integration](#5-xai)
6. [Training Pipeline](#6-training)
7. [Evaluation & Metrics](#7-evaluation)
8. [Visualizations](#8-visualizations)
9. [Conclusion](#9-conclusion)

## 1. Introduction & Mathematical Framework
<a id='1-introduction'></a>

### Key Innovation: Dynamic Prototype Refinement

Unlike static prototypical networks that compute prototypes once from support samples, **Dynamic Prototype Methods** iteratively refine prototypes by considering query information. This enables:

1. **Transductive Inference**: Query samples inform prototype computation
2. **Adaptive Prototypes**: Prototypes adapt to the current episode's distribution
3. **Attention-Based Aggregation**: Learn which support samples are most informative

### Mathematical Formulation

#### 1.1 Feature Encoding
Given an image $x_i$, extract features using CNN encoder:

$$h_i = f_\theta(x_i) \in \mathbb{R}^d$$

#### 1.2 Initial Prototype Computation
Standard mean-based prototype from support set:

$$c_k^{(0)} = \frac{1}{|\mathcal{S}_k|} \sum_{x_i \in \mathcal{S}_k} h_i$$

where $\mathcal{S}_k$ is the support set for class $k$.

#### 1.3 Dynamic Prototype Refinement (Key Innovation)
Iteratively refine prototypes using query-to-prototype relationships:

**Query-to-Prototype Similarity:**

$$\alpha_{qk} = \text{softmax}_k(s^T(q) \cdot c_k^{(t-1)})$$

**Query-to-Sample Attention:**

$$\beta_{ik} = \text{softmax}_i(\exp(-d(h_q, h_i))) \quad \text{for } x_i \in \mathcal{S}_k$$

**Prototype Update:**

$$c_k^{(t)} = \gamma_k \cdot c_k^{(t-1)} + (1 - \gamma_k) \cdot \sum_i \beta_{ik} \cdot h_i$$

where $t = 1, 2, \ldots, T$ (refinement steps) and $\gamma_k$ is a learnable decay parameter.

#### 1.4 Classification
After $T$ refinement steps, classify using negative distance:

$$P(y_q = k | x_q) = \text{softmax}_k(-d(h_q, c_k^{(T)}))$$

#### 1.5 Loss Function
Episode loss (negative log-probability):

$$\mathcal{L}_{episode} = -\sum_{q \in \mathcal{Q}} \log P(y_q | x_q, \mathcal{S})$$

In [ ]:
import os
import json
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
from scipy import stats
from scipy.stats import ttest_ind, wilcoxon
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, balanced_accuracy_score
)
from sklearn.calibration import calibration_curve

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

import torchvision
import torchvision.transforms as transforms
from torchvision import datasets

from captum.attr import Saliency, IntegratedGradients

import cv2

CONFIG = {
    'seed': 42,
    'data_root': './data',
    'output_dir': './dynamic_proto_output',
    'num_classes': 8,
    'images_per_class': 160,
    'train_ratio': 0.8,
    'val_ratio': 0.1,
    'test_ratio': 0.1,
    'n_way': 5,
    'k_shot': 5,
    'n_query': 15,
    'episodes': 1500,
    'hidden_dim': 256,
    'proto_dim': 128,
    'refinement_steps': 3,
    'dropout': 0.3,
    'lr': 0.001,
    'weight_decay': 1e-4,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'patience': 25,
}

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG['seed'])
print(f"Using device: {CONFIG['device']}")

## 2. Dataset Setup & Stratified Splitting
<a id='2-dataset'></a>

### Dataset Splitting Strategy

We perform an **80/10/10 stratified split** ensuring:
- **Training (80%)**: Used for episodic training
- **Validation (10%)**: Hyperparameter tuning and model selection
- **Testing (10%)**: Final evaluation

The stratification ensures class balance across all splits.

In [ ]:
class FSLDataset(Dataset):
    """Few-Shot Learning Dataset with stratified sampling."""

    def __init__(self, root_dir, split='train', transform=None, config=CONFIG):
        self.root_dir = Path(root_dir) / split
        self.split = split
        self.transform = transform
        self.config = config

        self.classes = sorted([d.name for d in self.root_dir.iterdir() if d.is_dir()])
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        self.idx_to_class = {idx: cls for cls, idx in self.class_to_idx.items()}

        self.samples = []
        for class_name in self.classes:
            class_dir = self.root_dir / class_name
            for img_path in class_dir.glob('*.jpg'):
                self.samples.append((str(img_path), self.class_to_idx[class_name]))
            for img_path in class_dir.glob('*.png'):
                self.samples.append((str(img_path), self.class_to_idx[class_name]))
            for img_path in class_dir.glob('*.jpeg'):
                self.samples.append((str(img_path), self.class_to_idx[class_name]))

        self.class_indices = defaultdict(list)
        for idx, (_, label) in enumerate(self.samples):
            self.class_indices[label].append(idx)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]

        image = torchvision.io.read_image(img_path)
        if image.shape[0] == 1:
            image = image.repeat(3, 1, 1)
        image = image.float() / 255.0

        if image.shape[1] != 84 or image.shape[2] != 84:
            image = F.interpolate(image.unsqueeze(0), size=(84, 84),
                                  mode='bilinear', align_corners=False).squeeze(0)

        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        image = (image - mean) / std

        if self.transform:
            image = self.transform(image)

        return image, label

    def get_class_samples(self, class_idx, num_samples=None):
        indices = self.class_indices[class_idx]
        if num_samples is not None:
            indices = random.sample(indices, min(num_samples, len(indices)))
        return [self.samples[i] for i in indices]


def create_stratified_split(root_dir, output_dir, config=CONFIG):
    """Create stratified train/val/test split."""
    import shutil
    
    output_dir = Path(output_dir)
    for split in ['train', 'val', 'test']:
        (output_dir / split).mkdir(parents=True, exist_ok=True)

    class_dirs = sorted([d for d in Path(root_dir).iterdir() if d.is_dir()])

    for class_dir in class_dirs:
        class_name = class_dir.name
        images = (list(class_dir.glob('*.jpg')) +
                  list(class_dir.glob('*.png')) +
                  list(class_dir.glob('*.jpeg')))
        random.shuffle(images)

        n_total = len(images)
        n_train = int(n_total * config['train_ratio'])
        n_val = int(n_total * config['val_ratio'])

        splits = {
            'train': images[:n_train],
            'val': images[n_train:n_train + n_val],
            'test': images[n_train + n_val:]
        }

        for split_name, split_images in splits.items():
            split_dir = output_dir / split_name / class_name
            split_dir.mkdir(parents=True, exist_ok=True)

            for img_path in split_images:
                dest_path = split_dir / img_path.name
                if not dest_path.exists():
                    shutil.copy(img_path, dest_path)

    print(f"Dataset split created in {output_dir}")
    return output_dir

## 3. Model Architecture
<a id='3-model'></a>

### Architecture Overview

```
┌─────────────────────────────────────────────────────────────────┐
│                Dynamic Prototype FSL Architecture              │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  ┌──────────────┐                                              │
│  │  Input Image │                                              │
│  └──────┬───────┘                                              │
│         │                                                      │
│         ▼                                                      │
│  ┌──────────────────────────────────────────────────────────┐  │
│  │              CNN Feature Encoder                         │  │
│  │         (5 Conv Blocks → 256 channels)                  │  │
│  └──────────────────────────────────────────────────────────┘  │
│         │                                                      │
│         ▼                                                      │
│  ┌──────────────────────────────────────────────────────────┐  │
│  │         Dynamic Prototype Refinement (×T steps)         │  │
│  │                                                         │  │
│  │   ┌──────────────────────────────────────────────────┐  │  │
│  │   │  1. Initial Prototype (mean of support set)      │  │  │
│  │   │  2. Compute query-to-support attention          │  │  │
│  │   │  3. Aggregate query-aware information           │  │  │
│  │   │  4. Update with learnable decay: c_k^(t)       │  │  │
│  │   └──────────────────────────────────────────────────┘  │  │
│  └──────────────────────────────────────────────────────────┘  │
│         │                                                      │
│         ▼                                                      │
│  ┌──────────────────────────────────────────────────────────┐  │
│  │              Classification                              │  │
│  │         P(y=k|x) = softmax(-d(h, c_k^(T)))               │  │
│  └──────────────────────────────────────────────────────────┘  │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
class ConvEncoder(nn.Module):
    """
    CNN Feature Encoder for Dynamic Prototype FSL.
    
    Architecture: 4 convolutional blocks with progressive channel increase.
    Output: 256-dimensional embedding
    """

    def __init__(self, in_channels=3, hidden_dim=256):
        super().__init__()

        self.conv_blocks = nn.Sequential(
            self._make_conv_block(in_channels, 64, kernel_size=3, padding=1),
            self._make_conv_block(64, 64, kernel_size=3, padding=1),
            self._make_conv_block(64, 128, kernel_size=3, padding=1),
            self._make_conv_block(128, 128, kernel_size=3, padding=1),
            self._make_conv_block(128, hidden_dim, kernel_size=3, padding=1),
            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.out_features = hidden_dim * 4 * 4

    def _make_conv_block(self, in_ch, out_ch, kernel_size=3, padding=1):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=kernel_size, padding=padding),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

    def forward(self, x):
        return self.conv_blocks(x).view(x.size(0), -1)


class PrototypeAttention(nn.Module):
    """
    Attention mechanism for dynamic prototype computation.
    
    Computes attention weights between query and support samples.
    a_ij = v^T · tanh(W · [h_i; h_j])
    """

    def __init__(self, feature_dim, hidden_dim=128):
        super().__init__()

        self.W = nn.Sequential(
            nn.Linear(feature_dim * 2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, support_embeddings, query_embeddings):
        """
        Args:
            support_embeddings: [n_support, feature_dim]
            query_embeddings: [n_query, feature_dim]
            
        Returns:
            attention_weights: [n_query, n_support] attention matrix
        """
        n_support = support_embeddings.size(0)
        n_query = query_embeddings.size(0)

        support_expanded = support_embeddings.unsqueeze(0).expand(n_query, -1, -1)
        query_expanded = query_embeddings.unsqueeze(1).expand(-1, n_support, -1)

        combined = torch.cat([support_expanded, query_expanded], dim=-1)
        attention_scores = self.W(combined).squeeze(-1)

        attention_weights = F.softmax(attention_scores, dim=-1)

        return attention_weights

## 4. Dynamic Prototype Refinement
<a id='4-dynamic-prototype'></a>

### Key Innovation: Iterative Prototype Refinement

The Dynamic Prototype Refinement module iteratively updates class prototypes by:

1. **Computing Attention**: Query-to-support attention weights
2. **Aggregating Information**: Weighted sum of support embeddings
3. **Gating Update**: Learnable combination of old and new prototypes

This process allows prototypes to adapt based on the current episode's query distribution.

In [ ]:
class DynamicPrototypeRefiner(nn.Module):
    """
    Dynamic Prototype Refinement Module.
    
    Key Innovation: Iteratively refines prototypes by considering query information.
    
    The refinement process:
    1. Start with initial prototype (mean of support embeddings)
    2. For each refinement step:
       - Compute attention weights between queries and supports
       - Aggregate query-aware information into prototypes
       - Update prototypes with learnable decay
    """

    def __init__(self, feature_dim, proto_dim=128, num_refinement_steps=3, dropout=0.3):
        super().__init__()

        self.feature_dim = feature_dim
        self.proto_dim = proto_dim
        self.num_steps = num_refinement_steps

        self.proto_projection = nn.Linear(feature_dim, proto_dim)
        self.attention = PrototypeAttention(proto_dim, hidden_dim=proto_dim // 2)

        self.update_net = nn.Sequential(
            nn.Linear(proto_dim * 2, proto_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(proto_dim, proto_dim)
        )

        self.gate_net = nn.Sequential(
            nn.Linear(proto_dim, proto_dim // 2),
            nn.ReLU(inplace=True),
            nn.Linear(proto_dim // 2, 1),
            nn.Sigmoid()
        )

        self.class_specific_decay = nn.Parameter(torch.ones(num_refinement_steps) * 0.5)

    def initial_prototype(self, support_embeddings, support_labels, n_way):
        """Compute initial prototype as class-wise mean."""
        prototypes = torch.zeros(n_way, support_embeddings.size(1), device=support_embeddings.device)

        for k in range(n_way):
            mask = (support_labels == k)
            if mask.sum() > 0:
                prototypes[k] = support_embeddings[mask].mean(0)

        return prototypes

    def refine_prototypes(self, prototypes, support_embeddings, support_labels,
                         query_embeddings, n_way, step):
        """
        Refine prototypes using query information.
        """
        projected_prototypes = self.proto_projection(prototypes)
        projected_support = self.proto_projection(support_embeddings)
        projected_query = self.proto_projection(query_embeddings)

        attention_weights = self.attention(projected_support, projected_query)

        refined_prototypes = []

        for k in range(n_way):
            proto = projected_prototypes[k]

            mask = (support_labels == k)
            class_support = projected_support[mask]

            if class_support.size(0) > 0:
                class_attention = attention_weights[:, mask]

                query_aware_support = (class_attention.unsqueeze(-1) * class_support.unsqueeze(0)).sum(1)
                query_aware_support = query_aware_support / (class_attention.sum(1, keepdim=True) + 1e-8)

                combined = torch.cat([proto, query_aware_support.mean(0)], dim=-1)
                update = self.update_net(combined)

                gate = self.gate_net(proto).squeeze(-1)
                decay = self.class_specific_decay[step].clamp(0.3, 0.9)

                new_proto = decay * proto + (1 - decay) * update

                refined_prototypes.append(new_proto)
            else:
                refined_prototypes.append(proto)

        return torch.stack(refined_prototypes)


class DynamicPrototypeClassifier(nn.Module):
    """
    Dynamic Prototype Classifier with iterative refinement.
    """

    def __init__(self, in_channels=3, hidden_dim=256, proto_dim=128,
                 num_refinement_steps=3, dropout=0.3, n_way=5):
        super().__init__()

        self.encoder = ConvEncoder(in_channels, hidden_dim)
        self.refiner = DynamicPrototypeRefiner(
            self.encoder.out_features, proto_dim, num_refinement_steps, dropout
        )

        self.classifier = nn.Sequential(
            nn.Linear(proto_dim, proto_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(proto_dim, n_way)
        )

        self.n_way = n_way

    def forward(self, support_images, query_images, support_labels):
        """
        Forward pass for one episode.
        """
        support_embeddings = self.encoder(support_images)
        query_embeddings = self.encoder(query_images)

        n_way = support_labels.unique().size(0)
        self.n_way = n_way

        prototypes = self.refiner.initial_prototype(support_embeddings, support_labels, n_way)

        all_prototypes = [prototypes]

        for step in range(self.refiner.num_steps):
            prototypes = self.refiner.refine_prototypes(
                prototypes, support_embeddings, support_labels,
                query_embeddings, n_way, step
            )
            all_prototypes.append(prototypes)

        distances = torch.cdist(query_embeddings, prototypes)
        logits = -distances

        return logits, prototypes, all_prototypes

    def get_embeddings(self, images):
        """Get embeddings for a set of images."""
        return self.encoder(images)

## 5. XAI Integration
<a id='5-xai'></a>

### Explainable AI for Dynamic Prototype FSL

We integrate multiple XAI techniques:

1. **Saliency Maps**: Gradient-based visualization showing input importance
2. **Integrated Gradients**: Path-integrated gradients for attribution
3. **Grad-CAM**: CNN-layer specific gradient visualization
4. **Prototype Evolution**: Visualize how prototypes change during refinement

### Mathematical Formulation

**Saliency Map:**
$$\text{Saliency}(x) = \frac{\partial f(x)_c}{\partial x}$$

**Integrated Gradients:**
$$\text{IG}_i(x) = (x_i - x'_i) \times \int_{\alpha=0}^{1} \frac{\partial F(x' + \alpha \times (x - x'))}{\partial x_i} d\alpha$$

In [ ]:
class XAIExplainer:
    """Explainable AI module for Dynamic Prototype FSL."""

    def __init__(self, model, device):
        self.model = model
        self.device = device
        self.saliency = Saliency(model)

    def get_saliency_map(self, images, target_class=None):
        """Compute gradient-based saliency maps."""
        self.model.eval()
        images = images.to(self.device).requires_grad_(True)

        if target_class is None:
            dummy_support = torch.zeros(5, *images.shape[1:], device=images.device)
            dummy_labels = torch.arange(5, device=images.device)
            with torch.no_grad():
                logits, _, _ = self.model(dummy_support, images[:len(images)], dummy_labels)
                target_class = logits[:len(images)].argmax(dim=1)

        saliency = self.saliency.attribute(images, target=target_class)
        return saliency.cpu().detach()

    def get_integrated_gradients(self, images, target_class=None, n_steps=50):
        """Compute Integrated Gradients attribution."""
        self.model.eval()
        images = images.to(self.device).requires_grad_(True)

        ig = IntegratedGradients(self.model)
        baseline = torch.zeros_like(images)

        if target_class is None:
            dummy_support = torch.zeros(5, *images.shape[1:], device=images.device)
            dummy_labels = torch.arange(5, device=images.device)
            with torch.no_grad():
                logits, _, _ = self.model(dummy_support, images[:len(images)], dummy_labels)
                target_class = logits[:len(images)].argmax(dim=1)

        attributions = ig.attribute(images, baseline=baseline, target=target_class, n_steps=n_steps)
        return attributions.cpu().detach()

    def generate_gradcam(self, images, target_layer=None, target_class=None):
        """Generate Grad-CAM visualization."""
        self.model.eval()
        images = images.to(self.device)

        if target_layer is None:
            target_layer = self.model.encoder.conv_blocks[-2]

        gradients, activations = [], []

        def backward_hook(module, grad_input, grad_output):
            gradients.append(grad_output[0])

        def forward_hook(module, input, output):
            activations.append(output)

        hooks = []
        hooks.append(target_layer.register_full_backward_hook(backward_hook))
        hooks.append(target_layer.register_forward_hook(forward_hook))

        images = images.clone().detach().requires_grad_(True)
        features = self.model.encoder.conv_blocks(images)

        dummy_support = torch.zeros(5, *images.shape[1:], device=images.device)
        dummy_labels = torch.arange(5, device=images.device)
        logits = self.model.classifier(features.flatten(1)[:1])

        if target_class is None:
            target_class = logits.argmax(dim=1)

        self.model.zero_grad()
        logits[0, target_class].backward()

        grad = gradients[0].cpu().detach()
        activation = activations[0].cpu().detach()

        for hook in hooks:
            hook.remove()

        weights = grad.mean(dim=(2, 3), keepdim=True)
        gradcam = (weights * activation).relu().squeeze().mean(dim=0)

        gradcam = F.interpolate(
            gradcam.unsqueeze(0).unsqueeze(0),
            size=(images.shape[2], images.shape[3]),
            mode='bilinear', align_corners=False
        ).squeeze()

        gradcam = (gradcam - gradcam.min()) / (gradcam.max() - gradcam.min() + 1e-8)

        return gradcam.numpy()

    def visualize_explanation(self, image, saliency, save_path=None):
        """Visualize saliency map with overlay."""
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))

        img_np = image.cpu().numpy().transpose(1, 2, 0)
        mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
        img_np = np.clip(img_np * std + mean, 0, 1)

        axes[0].imshow(img_np)
        axes[0].set_title('Original Image')
        axes[0].axis('off')

        saliency_np = saliency.squeeze().numpy()
        if saliency_np.ndim > 2:
            saliency_np = saliency_np.mean(axis=0)
        saliency_np = (saliency_np - saliency_np.min()) / (saliency_np.max() - saliency_np.min() + 1e-8)

        axes[1].imshow(saliency_np, cmap='jet')
        axes[1].set_title('Saliency Map')
        axes[1].axis('off')

        axes[2].imshow(img_np)
        axes[2].imshow(saliency_np, cmap='jet', alpha=0.6)
        axes[2].set_title('Overlay')
        axes[2].axis('off')

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            plt.close()

        return fig

    def visualize_prototype_evolution(self, all_prototypes, support_labels, n_way, save_path=None):
        """Visualize prototype evolution during refinement."""
        fig, axes = plt.subplots(1, len(all_prototypes), figsize=(4 * len(all_prototypes), 4))

        for step, prototypes in enumerate(all_prototypes):
            proto_np = prototypes.cpu().numpy()
            proto_2d = proto_np[:, :2] if proto_np.shape[1] >= 2 else proto_np

            for k in range(n_way):
                axes[step].scatter(proto_2d[k, 0], proto_2d[k, 1], s=100, label=f'Class {k}')

            axes[step].set_title(f'Prototype Step {step}')
            axes[step].set_xlabel('Dim 1')
            axes[step].set_ylabel('Dim 2')
            if step == 0:
                axes[step].legend()

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            plt.close()

        return fig

## 6. Training Pipeline
<a id='6-training'></a>

### Episodic Training Strategy

Dynamic Prototype FSL uses episodic training where each episode consists of:
- **N-way K-shot support set**: N classes with K examples each
- **Query set**: Unlabeled samples to classify

The model learns to:
1. Encode images into embeddings
2. Compute and iteratively refine class prototypes
3. Classify query samples based on prototype distance

In [ ]:
class EpisodicSampler:
    """Samples episodes for few-shot learning training."""

    def __init__(self, dataset, n_way=5, k_shot=5, n_query=15):
        self.dataset = dataset
        self.n_way = n_way
        self.k_shot = k_shot
        self.n_query = n_query
        self.class_indices = dataset.class_indices

    def sample_episode(self):
        """Sample a single episode."""
        available_classes = list(self.class_indices.keys())

        if len(available_classes) < self.n_way:
            selected_classes = available_classes
        else:
            selected_classes = random.sample(available_classes, self.n_way)

        support_images, support_labels = [], []
        query_images, query_labels = [], []

        for class_idx in selected_classes:
            class_samples = self.dataset.class_indices[class_idx]

            if len(class_samples) < self.k_shot + self.n_query:
                selected = random.sample(class_samples, len(class_samples))
            else:
                selected = random.sample(class_samples, self.k_shot + self.n_query)

            for idx in selected[:self.k_shot]:
                img, _ = self.dataset[idx]
                support_images.append(img)
                support_labels.append(class_idx)

            for idx in selected[self.k_shot:self.k_shot + self.n_query]:
                img, _ = self.dataset[idx]
                query_images.append(img)
                query_labels.append(class_idx)

        return (
            torch.stack(support_images), torch.tensor(support_labels, dtype=torch.long),
            torch.stack(query_images), torch.tensor(query_labels, dtype=torch.long),
            selected_classes
        )


class DynamicProtoTrainer:
    """Training pipeline for Dynamic Prototype FSL."""

    def __init__(self, model, optimizer, scheduler=None, device='cuda'):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device

        self.train_losses = []
        self.train_accs = []
        self.val_losses = []
        self.val_accs = []

        self.metrics_calc = MetricsCalculator()

    def train_episode(self, support_images, support_labels, query_images, query_labels):
        """Train on a single episode."""
        self.model.train()

        support_images = support_images.to(self.device)
        support_labels = support_labels.to(self.device)
        query_images = query_images.to(self.device)
        query_labels = query_labels.to(self.device)

        self.optimizer.zero_grad()

        query_logits, _, _ = self.model(support_images, query_images, support_labels)

        loss = F.cross_entropy(query_logits, query_labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
        self.optimizer.step()

        with torch.no_grad():
            preds = query_logits.argmax(dim=1)
            acc = (preds == query_labels).float().mean().item()

        return loss.item(), acc

    def evaluate(self, dataset, n_way=5, k_shot=5, n_query=15, n_episodes=100):
        """Evaluate the model on a dataset."""
        self.model.eval()

        all_preds, all_labels, all_probs = [], [], []

        sampler = EpisodicSampler(dataset, n_way, k_shot, n_query)

        with torch.no_grad():
            for _ in range(n_episodes):
                support_imgs, support_lbls, query_imgs, query_lbls, _ = sampler.sample_episode()

                support_imgs = support_imgs.to(self.device)
                support_lbls = support_lbls.to(self.device)
                query_imgs = query_imgs.to(self.device)

                query_logits, _, _ = self.model(support_imgs, query_imgs, support_lbls)

                probs = F.softmax(query_logits, dim=1).cpu().numpy()
                preds = query_logits.argmax(dim=1).cpu().numpy()
                labels = query_lbls.numpy()

                all_preds.extend(preds)
                all_labels.extend(labels)
                all_probs.extend(probs)

        return self.metrics_calc.compute_all_metrics(
            np.array(all_labels), np.array(all_preds), np.array(all_probs)
        )

    def train_full(self, train_dataset, val_dataset, n_episodes=1500,
                   n_way=5, k_shot=5, n_query=15, save_dir='./checkpoints'):
        """Full training loop with episodic training."""
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)

        best_val_acc = 0.0
        patience_counter = 0

        pbar = tqdm(range(n_episodes), desc='Training')

        for episode in pbar:
            sampler = EpisodicSampler(train_dataset, n_way, k_shot, n_query)
            support_imgs, support_lbls, query_imgs, query_lbls, _ = sampler.sample_episode()

            loss, acc = self.train_episode(support_imgs, support_lbls, query_imgs, query_lbls)

            self.train_losses.append(loss)
            self.train_accs.append(acc)

            if episode % 50 == 0:
                val_metrics = self.evaluate(val_dataset, n_episodes=50, n_way=n_way, k_shot=k_shot, n_query=n_query)
                val_acc = val_metrics['accuracy']
                self.val_losses.append(val_metrics.get('loss', loss))
                self.val_accs.append(val_acc)

                pbar.set_postfix({'loss': f'{loss:.4f}', 'acc': f'{acc:.4f}', 'val_acc': f'{val_acc:.4f}'})

                if val_acc > best_val_acc:
                    best_val_acc = val_acc
                    torch.save(self.model.state_dict(), save_dir / 'best_model.pth')
                    patience_counter = 0
                else:
                    patience_counter += 1

                if self.scheduler:
                    self.scheduler.step()

                if patience_counter >= CONFIG['patience']:
                    print(f"\nEarly stopping at episode {episode}")
                    break

        return self.train_losses, self.train_accs

## 7. Evaluation & Metrics
<a id='7-evaluation'></a>

### Metrics Overview

| Metric | Formula | Description |
|--------|---------|-------------|
| **Accuracy** | $\frac{TP + TN}{TP + TN + FP + FN}$ | Overall correctness |
| **F1-Score** | $\frac{2 \times Precision \times Recall}{Precision + Recall}$ | Harmonic mean |
| **ECE** | $\sum_b \frac{|B_b|}{n} |acc(B_b) - conf(B_b)|$ | Calibration error |
| **Sparsity** | $1 - \frac{non\_zero}{total}$ | Attribution sparsity |
| **t-test** | Statistical significance test | Compare model runs |

In [ ]:
class MetricsCalculator:
    """Computes evaluation metrics for Dynamic Prototype FSL."""

    def __init__(self, n_bins=15):
        self.n_bins = n_bins

    def compute_all_metrics(self, y_true, y_pred, y_prob=None, attributions=None):
        """Compute all evaluation metrics."""
        metrics = {}

        metrics['accuracy'] = accuracy_score(y_true, y_pred)
        metrics['balanced_accuracy'] = balanced_accuracy_score(y_true, y_pred)

        metrics['f1_macro'] = f1_score(y_true, y_pred, average='macro', zero_division=0)
        metrics['f1_micro'] = f1_score(y_true, y_pred, average='micro', zero_division=0)
        metrics['f1_weighted'] = f1_score(y_true, y_pred, average='weighted', zero_division=0)

        metrics['precision_macro'] = precision_score(y_true, y_pred, average='macro', zero_division=0)
        metrics['recall_macro'] = recall_score(y_true, y_pred, average='macro', zero_division=0)

        if y_prob is not None:
            metrics['ece'] = self._compute_ece(y_true, y_pred, y_prob)

        if attributions is not None:
            metrics['attribution_sparsity'] = self._compute_sparsity(attributions)

        return metrics

    def _compute_ece(self, y_true, y_pred, y_prob, n_bins=15):
        """Compute Expected Calibration Error."""
        confidences = np.max(y_prob, axis=1)
        accuracies = (y_pred == y_true).astype(float)

        bin_edges = np.linspace(0, 1, n_bins + 1)
        ece = 0.0

        for i in range(n_bins):
            bin_mask = (confidences >= bin_edges[i]) & (confidences < bin_edges[i + 1])
            if bin_mask.sum() > 0:
                bin_acc = accuracies[bin_mask].mean()
                bin_conf = confidences[bin_mask].mean()
                ece += (bin_mask.sum() / len(y_true)) * abs(bin_acc - bin_conf)

        return ece

    def _compute_sparsity(self, attributions):
        """Compute attribution sparsity."""
        if attributions is None:
            return None

        attributions = np.abs(attributions)
        threshold = attributions.max() * 0.01
        non_zero = (attributions > threshold).sum()
        sparsity = 1 - (non_zero / attributions.size)

        return sparsity

    def compute_per_class_metrics(self, y_true, y_pred, n_classes):
        """Compute per-class precision, recall, F1."""
        precision = precision_score(y_true, y_pred, average=None, zero_division=0)
        recall = recall_score(y_true, y_pred, average=None, zero_division=0)
        f1 = f1_score(y_true, y_pred, average=None, zero_division=0)

        return {
            'precision_per_class': precision.tolist(),
            'recall_per_class': recall.tolist(),
            'f1_per_class': f1.tolist()
        }

    def statistical_test(self, scores1, scores2, method='ttest'):
        """Perform statistical significance test."""
        if method == 'ttest':
            stat, p_value = ttest_ind(scores1, scores2)
        else:
            stat, p_value = wilcoxon(scores1, scores2)

        return {'statistic': stat, 'p_value': p_value, 'significant': p_value < 0.05}

## 8. Visualizations
<a id='8-visualizations'></a>

### Visualization Suite

We generate comprehensive visualizations:

1. **Training History**: Loss and accuracy curves
2. **Confusion Matrix**: Per-class performance heatmap
3. **Per-Class Metrics**: Precision, Recall, F1 bar plots
4. **Calibration Curve**: Reliability diagram
5. **XAI Visualizations**: Saliency maps and Grad-CAM
6. **Prototype Evolution**: How prototypes change during refinement

In [ ]:
class Visualizer:
    """Visualization utilities for Dynamic Prototype FSL."""

    def __init__(self, output_dir='./visualizations', class_names=None):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.class_names = class_names or [f'Class_{i}' for i in range(8)]
        self.xai_explainer = None

    def plot_training_history(self, train_losses, train_accs, val_losses, val_accs):
        """Plot training and validation curves."""
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        episodes = range(len(train_losses))
        val_episodes = range(0, len(train_losses), 50)

        axes[0].plot(episodes, train_losses, 'b-', alpha=0.5, label='Training Loss')
        if len(val_losses) == len(val_episodes):
            axes[0].plot(list(val_episodes), val_losses, 'r-', linewidth=2, label='Validation Loss')
        axes[0].set_xlabel('Episode')
        axes[0].set_ylabel('Loss')
        axes[0].set_title('Training and Validation Loss')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        axes[1].plot(episodes, train_accs, 'b-', alpha=0.5, label='Training Accuracy')
        if len(val_accs) == len(val_episodes):
            axes[1].plot(list(val_episodes), val_accs, 'r-', linewidth=2, label='Validation Accuracy')
        axes[1].set_xlabel('Episode')
        axes[1].set_ylabel('Accuracy')
        axes[1].set_title('Training and Validation Accuracy')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(self.output_dir / 'training_history.png', dpi=150, bbox_inches='tight')
        plt.show()
        plt.close()

        return fig

    def plot_confusion_matrix(self, y_true, y_pred, class_names=None):
        """Plot normalized confusion matrix."""
        if class_names is None:
            class_names = self.class_names[:len(np.unique(y_true)) + 1]

        cm = confusion_matrix(y_true, y_pred)
        cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

        fig, ax = plt.subplots(figsize=(10, 8))

        sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues',
                   xticklabels=class_names, yticklabels=class_names, ax=ax,
                   cbar_kws={'label': 'Proportion'})

        ax.set_xlabel('Predicted Label')
        ax.set_ylabel('True Label')
        ax.set_title('Normalized Confusion Matrix - Dynamic Prototype FSL')

        plt.tight_layout()
        plt.savefig(self.output_dir / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
        plt.show()
        plt.close()

        return fig

    def plot_per_class_metrics(self, metrics, class_names=None):
        """Plot per-class precision, recall, F1 scores."""
        if class_names is None:
            class_names = self.class_names

        n_classes = len(class_names)
        precision = metrics.get('precision_per_class', [0] * n_classes)
        recall = metrics.get('recall_per_class', [0] * n_classes)
        f1 = metrics.get('f1_per_class', [0] * n_classes)

        x = np.arange(n_classes)
        width = 0.25

        fig, ax = plt.subplots(figsize=(12, 6))

        ax.bar(x - width, precision[:n_classes], width, label='Precision', color='#2ecc71')
        ax.bar(x, recall[:n_classes], width, label='Recall', color='#3498db')
        ax.bar(x + width, f1[:n_classes], width, label='F1-Score', color='#e74c3c')

        ax.set_xlabel('Class')
        ax.set_ylabel('Score')
        ax.set_title('Per-Class Performance Metrics')
        ax.set_xticks(x)
        ax.set_xticklabels(class_names[:n_classes], rotation=45, ha='right')
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        ax.set_ylim([0, 1.1])

        plt.tight_layout()
        plt.savefig(self.output_dir / 'per_class_metrics.png', dpi=150, bbox_inches='tight')
        plt.show()
        plt.close()

        return fig

    def plot_calibration_curve(self, y_true, y_prob, n_bins=10):
        """Plot reliability diagram for calibration."""
        conf_true = np.max(y_prob, axis=1)
        conf_pred = y_true == y_prob.argmax(axis=1)

        fig, ax = plt.subplots(figsize=(8, 8))

        ax.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')

        prob_true, prob_pred = calibration_curve(conf_pred, conf_true, n_bins=n_bins)

        ax.plot(prob_pred, prob_true, 'o-', color='#3498db', linewidth=2, markersize=8, label='Dynamic Proto FSL')

        ax.set_xlabel('Mean Predicted Probability')
        ax.set_ylabel('Fraction of Positives')
        ax.set_title('Calibration Curve (Reliability Diagram)')
        ax.legend(loc='lower right')
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(self.output_dir / 'calibration_curve.png', dpi=150, bbox_inches='tight')
        plt.show()
        plt.close()

        return fig

## 9. Main Experiment Runner
<a id='9-main'></a>

In [ ]:
def run_experiment(data_root='./data', output_dir='./dynamic_proto_output', config=CONFIG):
    """Run complete Dynamic Prototype FSL experiment pipeline."""
    print("=" * 60)
    print("Dynamic Prototype FSL: Few-Shot Learning with Dynamic Refinement")
    print("=" * 60)
    print(f"Device: {config['device']}")
    print(f"Output Directory: {output_dir}")
    print()

    output_dir = Path(output_dir)
    visualizer = Visualizer(output_dir / 'visualizations')

    print("Step 1: Loading datasets...")
    transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    ])

    train_dataset = FSLDataset(data_root, split='train', transform=transform)
    val_dataset = FSLDataset(data_root, split='val')
    test_dataset = FSLDataset(data_root, split='test')

    print(f"Train samples: {len(train_dataset)}")
    print(f"Val samples: {len(val_dataset)}")
    print(f"Test samples: {len(test_dataset)}")
    print(f"Classes: {train_dataset.classes}")
    print()

    print("Step 2: Initializing model...")
    model = DynamicPrototypeClassifier(
        in_channels=3,
        hidden_dim=config['hidden_dim'],
        proto_dim=config['proto_dim'],
        num_refinement_steps=config['refinement_steps'],
        dropout=config['dropout'],
        n_way=config['n_way']
    )

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print()

    print("Step 3: Setting up training...")
    optimizer = Adam(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    scheduler = CosineAnnealingLR(optimizer, T_max=config['episodes'], eta_min=1e-6)

    trainer = DynamicProtoTrainer(model, optimizer, scheduler, device=config['device'])

    print("Step 4: Training model...")
    trainer.train_full(
        train_dataset, val_dataset,
        n_episodes=config['episodes'],
        n_way=config['n_way'],
        k_shot=config['k_shot'],
        n_query=config['n_query'],
        save_dir=output_dir / 'checkpoints'
    )
    print()

    print("Step 5: Evaluating model...")

    test_metrics = trainer.evaluate(
        test_dataset,
        n_episodes=200,
        n_way=config['n_way'],
        k_shot=config['k_shot'],
        n_query=config['n_query']
    )

    print("\nTest Set Metrics:")
    print(f"  Accuracy: {test_metrics['accuracy']:.4f}")
    print(f"  F1 (Macro): {test_metrics['f1_macro']:.4f}")
    print(f"  F1 (Micro): {test_metrics['f1_micro']:.4f}")
    print(f"  F1 (Weighted): {test_metrics['f1_weighted']:.4f}")
    print(f"  Balanced Accuracy: {test_metrics['balanced_accuracy']:.4f}")
    if 'ece' in test_metrics:
        print(f"  ECE: {test_metrics['ece']:.4f}")
    if 'attribution_sparsity' in test_metrics:
        print(f"  Attribution Sparsity: {test_metrics['attribution_sparsity']:.4f}")
    print()

    print("Step 6: Generating visualizations...")

    visualizer.plot_training_history(
        trainer.train_losses, trainer.train_accs,
        trainer.val_losses, trainer.val_accs
    )

    sampler = EpisodicSampler(test_dataset, config['n_way'], config['k_shot'], config['n_query'])
    support_imgs, support_lbls, query_imgs, query_lbls, selected_classes = sampler.sample_episode()

    all_preds, all_labels, all_probs = [], [], []

    for _ in range(10):
        s_imgs, s_lbls, q_imgs, q_lbls, _ = sampler.sample_episode()
        with torch.no_grad():
            q_logits, _, _ = model(
                s_imgs.to(config['device']),
                q_imgs.to(config['device']),
                s_lbls.to(config['device'])
            )
        all_preds.extend(q_logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(q_lbls.numpy())
        all_probs.extend(F.softmax(q_logits, dim=1).cpu().numpy())

    class_names = [train_dataset.idx_to_class[i] for i in sorted(selected_classes)]
    visualizer.plot_confusion_matrix(np.array(all_labels), np.array(all_preds), class_names)

    print("Step 7: Generating XAI visualizations...")

    xai = XAIExplainer(model, config['device'])
    visualizer.xai_explainer = xai

    sample_images = query_imgs[:5]

    for i, img in enumerate(sample_images):
        saliency = xai.get_saliency_map(img.unsqueeze(0))
        xai.visualize_explanation(img, saliency[0], save_path=visualizer.output_dir / f'xai_saliency_{i}.png')

    print("\n" + "=" * 60)
    print("Dynamic Prototype FSL Experiment Complete!")
    print("=" * 60)
    print(f"Results saved to: {output_dir}")

    results = {
        'config': config,
        'test_metrics': test_metrics,
        'total_params': total_params,
        'trainable_params': trainable_params
    }

    with open(output_dir / 'results.json', 'w') as f:
        json.dump(results, f, indent=2, default=str)

    return model, results

# Run experiment
# model, results = run_experiment()

## 9. Conclusion
<a id='9-conclusion'></a>

### Summary

This notebook implemented a **Dynamic Prototype FSL** model with comprehensive XAI integration:

#### Key Contributions:

1. **Dynamic Prototype Refinement**: Iteratively updates prototypes using query information

2. **Attention Mechanism**: Learns query-to-support attention for adaptive aggregation

3. **Learnable Decay**: Class-specific decay parameters control prototype update rate

4. **XAI Integration**: Multiple explanation techniques for model interpretability

5. **Comprehensive Metrics**: Accuracy, F1, ECE, Attribution Sparsity, and statistical tests

#### Comparison with Other FSL Methods:

| Method | Prototype Computation | Query Adaptation |
|--------|---------------------|-------------------|
| **Prototypical Networks** | Static mean | None |
| **Siamese Networks** | Pairwise comparison | None |
| **Relation Networks** | Learned comparison | None |
| **GNN-FSL** | Graph-attended | Graph structure |
| **Dynamic Proto FSL** | Iteratively refined | Query-aware updates |

#### Future Improvements:

- Multi-level prototype refinement (coarse-to-fine)
- Cross-episode prototype memory
- Meta-learning for refinement parameters
- SHAP-based explanations